In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

uploaded = files.upload()

In [ ]:
# V9 Cell 1 — Load and Audit 2024 US Open Dataset

DATA_PATH = next(iter(uploaded))

usopen_df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("V9 — US OPEN EMPIRICAL ANALYSIS")
print("=" * 70)

print("\nDataset shape:")
print(usopen_df.shape)

print("\nNumber of columns:")
print(len(usopen_df.columns))

print("\nKey serve columns present:")

key_columns = [
    "PointNumber",
    "ServeNumber",
    "Speed_KMH"
]

for col in key_columns:
    print(
        f"{col:20s}: "
        f"{'YES' if col in usopen_df.columns else 'NO'}"
    )

print("\n" + "=" * 70)
print("V9 DATASET LOAD COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 2 — Identify Match, Player, and Sex Fields

print("=" * 70)
print("V9 — DATASET FIELD AUDIT")
print("=" * 70)

print("\nAll dataset columns:")
for i, col in enumerate(usopen_df.columns):
    print(f"{i:2d}: {col}")

print("\n" + "=" * 70)

print("Candidate identity / grouping columns:")
identity_keywords = [
    "match",
    "player",
    "winner",
    "loser",
    "gender",
    "sex",
    "server"
]

for col in usopen_df.columns:
    if any(keyword in col.lower() for keyword in identity_keywords):
        print(col)

print("\n" + "=" * 70)
print("Sample rows — identity and serve information:")

In [ ]:
# V9 Cell 3 — Inspect Match Metadata Rows

print("=" * 70)
print("V9 — MATCH METADATA INSPECTION")
print("=" * 70)

# Metadata rows identified during the original dataset audit
metadata_df = usopen_df[
    usopen_df["PointNumber"].astype(str).str.match(r"0[X,Y]")
].copy()

print("\nMetadata rows:")
print(f"Count: {len(metadata_df)}")

print("\nUnique PointNumber values:")
print(metadata_df["PointNumber"].value_counts().to_string())

print("\nSelected metadata fields:")

metadata_columns = [
    "match_id",
    "PointNumber",
    "PointServer",
    "Speed_KMH",
    "ServeNumber",
    "ServeIndicator",
    "ServingTo",
    "Serve_Direction",
    "ServeWidth",
    "ServeDepth",
    "ReturnDepth"
]

metadata_columns = [
    col for col in metadata_columns
    if col in metadata_df.columns
]

display(
    metadata_df[metadata_columns].head(20)
)

print("\n" + "=" * 70)
print("V9 MATCH METADATA INSPECTION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 4 — Verify Server Encoding and Serve Indicators

print("=" * 70)
print("V9 — SERVER / SERVE-INDICATOR VALIDATION")
print("=" * 70)

# Remove the 0X / 0Y metadata rows
actual_points = usopen_df[
    ~usopen_df["PointNumber"].astype(str).str.match(r"0[X,Y]")
].copy()

print("\nActual point rows:")
print(len(actual_points))

print("\nPointServer values:")
print(
    actual_points["PointServer"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nServeNumber values:")
print(
    actual_points["ServeNumber"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nServeIndicator values:")
print(
    actual_points["ServeIndicator"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n" + "=" * 70)
print("P1 SERVE INDICATORS")
print("=" * 70)

p1_cols = [
    "P1FirstSrvIn",
    "P1FirstSrvWon",
    "P1SecondSrvIn",
    "P1SecondSrvWon",
    "P1DoubleFault"
]

display(
    actual_points[p1_cols]
    .value_counts(dropna=False)
    .head(15)
)

print("\n" + "=" * 70)
print("P2 SERVE INDICATORS")
print("=" * 70)

p2_cols = [
    "P2FirstSrvIn",
    "P2FirstSrvWon",
    "P2SecondSrvIn",
    "P2SecondSrvWon",
    "P2DoubleFault"
]

display(
    actual_points[p2_cols]
    .value_counts(dropna=False)
    .head(15)
)

print("\n" + "=" * 70)
print("V9 SERVER / SERVE-INDICATOR VALIDATION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 5 — Inspect Actual Serve-Level Records

print("=" * 70)
print("V9 — SERVE RECORD INSPECTION")
print("=" * 70)

serve_cols = [
    "match_id",
    "PointNumber",
    "PointServer",
    "ServeIndicator",
    "ServeNumber",
    "Speed_KMH",
    "P1FirstSrvIn",
    "P2FirstSrvIn",
    "P1FirstSrvWon",
    "P2FirstSrvWon",
    "P1SecondSrvIn",
    "P2SecondSrvIn",
    "P1SecondSrvWon",
    "P2SecondSrvWon",
    "P1DoubleFault",
    "P2DoubleFault"
]

# ------------------------------------------------------------
# One example of each serve-number category
# ------------------------------------------------------------

for serve_number in [0, 1, 2]:

    print("\n" + "=" * 70)
    print(f"SERVE NUMBER = {serve_number}")
    print("=" * 70)

    sample = actual_points[
        actual_points["ServeNumber"] == serve_number
    ][serve_cols].head(10)

    display(sample)

# ------------------------------------------------------------
# Inspect the 9 PointServer / ServeIndicator disagreements
# ------------------------------------------------------------

disagreements = actual_points[
    actual_points["PointServer"] != actual_points["ServeIndicator"]
].copy()

print("\n" + "=" * 70)
print("POINTSERVER / SERVEINDICATOR DISAGREEMENTS")
print("=" * 70)

print(f"\nNumber of disagreements: {len(disagreements)}")

display(
    disagreements[serve_cols]
)

In [ ]:
# V9 Cell 6 — Validate Serve Outcomes Against PointServer

print("=" * 70)
print("V9 — SERVE OUTCOME VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# First serves
# ------------------------------------------------------------

first_serves = actual_points[
    actual_points["ServeNumber"] == 1
].copy()

# A first serve is recorded for the actual server's P1/P2 field.
first_serves["server_first_srv_in"] = np.where(
    first_serves["PointServer"] == 1,
    first_serves["P1FirstSrvIn"],
    first_serves["P2FirstSrvIn"]
)

first_serves["server_first_srv_won"] = np.where(
    first_serves["PointServer"] == 1,
    first_serves["P1FirstSrvWon"],
    first_serves["P2FirstSrvWon"]
)

print("\nFIRST SERVES")
print("-" * 70)

print("Total first-serve rows:", len(first_serves))

print("\nServer's FirstSrvIn values:")
print(
    first_serves["server_first_srv_in"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nServer's FirstSrvWon values:")
print(
    first_serves["server_first_srv_won"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# Second serves
# ------------------------------------------------------------

second_serves = actual_points[
    actual_points["ServeNumber"] == 2
].copy()

second_serves["server_second_srv_in"] = np.where(
    second_serves["PointServer"] == 1,
    second_serves["P1SecondSrvIn"],
    second_serves["P2SecondSrvIn"]
)

second_serves["server_second_srv_won"] = np.where(
    second_serves["PointServer"] == 1,
    second_serves["P1SecondSrvWon"],
    second_serves["P2SecondSrvWon"]
)

print("\nSECOND SERVES")
print("-" * 70)

print("Total second-serve rows:", len(second_serves))

print("\nServer's SecondSrvIn values:")
print(
    second_serves["server_second_srv_in"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nServer's SecondSrvWon values:")
print(
    second_serves["server_second_srv_won"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# Double faults
# ------------------------------------------------------------

double_faults = actual_points[
    actual_points["ServeNumber"] == 0
].copy()

double_faults["server_double_fault"] = np.where(
    double_faults["PointServer"] == 1,
    double_faults["P1DoubleFault"],
    double_faults["P2DoubleFault"]
)

print("\nDOUBLE FAULTS")
print("-" * 70)

print("ServeNumber = 0 rows:", len(double_faults))

print("\nServer's DoubleFault values:")
print(
    double_faults["server_double_fault"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n" + "=" * 70)
print("V9 SERVE OUTCOME VALIDATION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 7 — Validate Serve Sequence Structure

print("=" * 70)
print("V9 — SERVE SEQUENCE STRUCTURE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Work within each match and preserve original row order.
# ------------------------------------------------------------

sequence_df = actual_points.copy()

sequence_df["row_order"] = np.arange(len(sequence_df))

# ------------------------------------------------------------
# For each point, inspect the relationship between:
#   current PointNumber
#   PointServer
#   ServeNumber
#   Speed_KMH
#
# We will examine points where ServeNumber = 2 and compare
# them with the immediately preceding row.
# ------------------------------------------------------------

second_rows = sequence_df[
    sequence_df["ServeNumber"] == 2
].copy()

second_rows["previous_server"] = (
    sequence_df["PointServer"]
    .shift(1)
    .loc[second_rows.index]
)

second_rows["previous_serve_number"] = (
    sequence_df["ServeNumber"]
    .shift(1)
    .loc[second_rows.index]
)

second_rows["previous_speed"] = (
    sequence_df["Speed_KMH"]
    .shift(1)
    .loc[second_rows.index]
)

# ------------------------------------------------------------
# Check whether second serves follow a first-serve record
# from the same server within the same match.
# ------------------------------------------------------------

second_rows["same_server_as_previous"] = (
    second_rows["PointServer"]
    == second_rows["previous_server"]
)

second_rows["previous_was_first_serve"] = (
    second_rows["previous_serve_number"] == 1
)

print("\nTotal second-serve rows:")
print(len(second_rows))

print("\nSecond serves preceded by same-server first-serve row:")
print(
    second_rows["same_server_as_previous"]
    .value_counts(dropna=False)
)

print("\nSecond serves preceded by ServeNumber = 1:")
print(
    second_rows["previous_was_first_serve"]
    .value_counts(dropna=False)
)

print("\nBoth conditions satisfied:")
print(
    (
        second_rows["same_server_as_previous"]
        & second_rows["previous_was_first_serve"]
    ).sum()
)

print("\nExample second-serve sequences:")

display(
    sequence_df[
        sequence_df["match_id"] == "2024-usopen-1101"
    ][
        [
            "PointNumber",
            "PointServer",
            "ServeNumber",
            "Speed_KMH"
        ]
    ].head(30)
)

print("\n" + "=" * 70)
print("V9 SERVE SEQUENCE VALIDATION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 8 — Examine ServeNumber Semantics

print("=" * 70)
print("V9 — SERVENUMBER SEMANTICS")
print("=" * 70)

# ------------------------------------------------------------
# Basic statistics by ServeNumber
# ------------------------------------------------------------

serve_summary = actual_points.groupby("ServeNumber").agg(
    rows=("ServeNumber", "size"),
    speed_nonzero=("Speed_KMH", lambda x: (x > 0).sum()),
    speed_zero=("Speed_KMH", lambda x: (x == 0).sum()),
    speed_missing=("Speed_KMH", lambda x: x.isna().sum()),
    mean_speed=("Speed_KMH", lambda x: x[x > 0].mean()),
    median_speed=("Speed_KMH", lambda x: x[x > 0].median()),
)

print("\nServeNumber summary:")
display(serve_summary)

# ------------------------------------------------------------
# WinnerType by ServeNumber
# ------------------------------------------------------------

print("\nWinnerType distribution by ServeNumber:")
winner_type_table = pd.crosstab(
    actual_points["ServeNumber"],
    actual_points["WinnerType"],
    dropna=False,
    normalize="index"
) * 100

display(winner_type_table.round(2))

# ------------------------------------------------------------
# WinnerShotType by ServeNumber
# ------------------------------------------------------------

print("\nWinnerShotType distribution by ServeNumber:")
winner_shot_table = pd.crosstab(
    actual_points["ServeNumber"],
    actual_points["WinnerShotType"],
    dropna=False,
    normalize="index"
) * 100

display(winner_shot_table.round(2))

# ------------------------------------------------------------
# PointWinner relative to PointServer
# ------------------------------------------------------------

point_result = actual_points.copy()

point_result["server_won_point"] = (
    point_result["PointWinner"] == point_result["PointServer"]
)

print("\nPoint outcome by ServeNumber:")
point_win_table = point_result.groupby("ServeNumber").agg(
    points=("ServeNumber", "size"),
    server_won=("server_won_point", "sum"),
    server_win_pct=("server_won_point", "mean")
)

point_win_table["server_win_pct"] *= 100

display(point_win_table.round(2))

print("\n" + "=" * 70)
print("V9 SERVENUMBER SEMANTICS COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 9 — Construct Clean Empirical Serve Samples

print("=" * 70)
print("V9 — CLEAN EMPIRICAL SERVE DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Identify recorded serve observations
# ------------------------------------------------------------

first_serves = actual_points[
    actual_points["ServeNumber"] == 1
].copy()

second_serves = actual_points[
    actual_points["ServeNumber"] == 2
].copy()

double_faults = actual_points[
    actual_points["ServeNumber"] == 0
].copy()

# ------------------------------------------------------------
# Separate valid observed speeds from zero-speed records.
# Zero is treated as missing/censored for speed analysis.
# ------------------------------------------------------------

first_valid = first_serves[
    first_serves["Speed_KMH"] > 0
].copy()

second_valid = second_serves[
    second_serves["Speed_KMH"] > 0
].copy()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary = pd.DataFrame({
    "Serve type": [
        "First serve",
        "Second serve",
        "Double fault"
    ],
    "Total records": [
        len(first_serves),
        len(second_serves),
        len(double_faults)
    ],
    "Valid speed records": [
        len(first_valid),
        len(second_valid),
        0
    ],
    "Zero-speed records": [
        (first_serves["Speed_KMH"] == 0).sum(),
        (second_serves["Speed_KMH"] == 0).sum(),
        (double_faults["Speed_KMH"] == 0).sum()
    ]
})

print("\nClean empirical dataset summary:")
display(summary)

# ------------------------------------------------------------
# Speed summaries
# ------------------------------------------------------------

speed_summary = pd.DataFrame({
    "Serve type": [
        "First serve",
        "Second serve"
    ],
    "N": [
        len(first_valid),
        len(second_valid)
    ],
    "Mean (km/h)": [
        first_valid["Speed_KMH"].mean(),
        second_valid["Speed_KMH"].mean()
    ],
    "Median (km/h)": [
        first_valid["Speed_KMH"].median(),
        second_valid["Speed_KMH"].median()
    ],
    "SD (km/h)": [
        first_valid["Speed_KMH"].std(),
        second_valid["Speed_KMH"].std()
    ],
    "Min (km/h)": [
        first_valid["Speed_KMH"].min(),
        second_valid["Speed_KMH"].min()
    ],
    "Max (km/h)": [
        first_valid["Speed_KMH"].max(),
        second_valid["Speed_KMH"].max()
    ]
})

print("\nObserved serve-speed summary:")
display(speed_summary.round(2))

# ------------------------------------------------------------
# Store a combined analysis dataset
# ------------------------------------------------------------

first_valid["serve_type"] = "First"
second_valid["serve_type"] = "Second"

empirical_serves = pd.concat(
    [
        first_valid,
        second_valid
    ],
    ignore_index=True
)

print("\nCombined valid-speed observations:", len(empirical_serves))

print("\nServe type counts:")
print(empirical_serves["serve_type"].value_counts())

print("\n" + "=" * 70)
print("V9 CLEAN EMPIRICAL SERVE DATASET COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 10 — Empirical Speeds vs Computational Speed Range

print("=" * 70)
print("V9 — EMPIRICAL SPEEDS VS COMPUTATIONAL RANGE")
print("=" * 70)

# ------------------------------------------------------------
# Define computational speed range from V6
# ------------------------------------------------------------

MODEL_MIN_SPEED = 160.0
MODEL_MAX_SPEED = 220.0

# ------------------------------------------------------------
# Function to summarize empirical speeds
# ------------------------------------------------------------

def summarize_speed_range(series, label):

    s = series.dropna()

    below = (s < MODEL_MIN_SPEED).sum()
    inside = ((s >= MODEL_MIN_SPEED) & (s <= MODEL_MAX_SPEED)).sum()
    above = (s > MODEL_MAX_SPEED).sum()

    summary = {
        "Serve type": label,
        "N": len(s),
        "Below 160": below,
        "160–220": inside,
        "Above 220": above,
        "% Below 160": 100 * below / len(s),
        "% 160–220": 100 * inside / len(s),
        "% Above 220": 100 * above / len(s),
    }

    return summary


range_summary = pd.DataFrame([
    summarize_speed_range(
        first_valid["Speed_KMH"],
        "First serve"
    ),
    summarize_speed_range(
        second_valid["Speed_KMH"],
        "Second serve"
    )
])

print("\nObserved speeds relative to V6 computational range:")
display(range_summary.round(2))

# ------------------------------------------------------------
# Overall distribution statistics
# ------------------------------------------------------------

print("\nOverall empirical speed distribution:")

overall_speed = empirical_serves["Speed_KMH"]

print("Minimum:", overall_speed.min())
print("5th percentile:", overall_speed.quantile(0.05))
print("25th percentile:", overall_speed.quantile(0.25))
print("Median:", overall_speed.median())
print("75th percentile:", overall_speed.quantile(0.75))
print("95th percentile:", overall_speed.quantile(0.95))
print("Maximum:", overall_speed.max())

print("\n" + "=" * 70)
print("V9 EMPIRICAL SPEED RANGE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 11 — Empirical Serve-Speed Distribution

print("=" * 70)
print("V9 — EMPIRICAL SERVE-SPEED DISTRIBUTION")
print("=" * 70)

plt.figure(figsize=(10, 6))

plt.hist(
    first_valid["Speed_KMH"],
    bins=np.arange(90, 236, 5),
    alpha=0.6,
    label="First serve",
    density=True
)

plt.hist(
    second_valid["Speed_KMH"],
    bins=np.arange(90, 236, 5),
    alpha=0.6,
    label="Second serve",
    density=True
)

# Computational envelope range
plt.axvline(
    MODEL_MIN_SPEED,
    linestyle="--",
    linewidth=1.5,
    label="Model range: 160–220 km/h"
)

plt.axvline(
    MODEL_MAX_SPEED,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Serve speed (km/h)")
plt.ylabel("Density")
plt.title("2024 US Open Observed Serve-Speed Distributions")
plt.legend()
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

print("\nDistribution plotted successfully.")

print("\n" + "=" * 70)
print("V9 EMPIRICAL SPEED DISTRIBUTION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 12 — First-Serve Speed-Bin Analysis

print("=" * 70)
print("V9 — FIRST-SERVE SPEED-BIN ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Create speed bins
# ------------------------------------------------------------

first_analysis = first_valid.copy()

bins = np.arange(90, 241, 10)

first_analysis["speed_bin"] = pd.cut(
    first_analysis["Speed_KMH"],
    bins=bins,
    right=False
)

# ------------------------------------------------------------
# Determine whether server won the point
# ------------------------------------------------------------

first_analysis["server_won_point"] = (
    first_analysis["PointWinner"] == first_analysis["PointServer"]
)

# ------------------------------------------------------------
# Aggregate by speed bin
# ------------------------------------------------------------

speed_bin_summary = (
    first_analysis
    .groupby("speed_bin", observed=False)
    .agg(
        observations=("Speed_KMH", "size"),
        mean_speed=("Speed_KMH", "mean"),
        median_speed=("Speed_KMH", "median"),
        server_wins=("server_won_point", "sum"),
        server_win_pct=("server_won_point", "mean")
    )
    .reset_index()
)

speed_bin_summary["share_of_first_serves_pct"] = (
    100
    * speed_bin_summary["observations"]
    / len(first_analysis)
)

speed_bin_summary["server_win_pct"] *= 100

# ------------------------------------------------------------
# Remove empty bins
# ------------------------------------------------------------

speed_bin_summary = speed_bin_summary[
    speed_bin_summary["observations"] > 0
].copy()

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nFirst-serve speed-bin summary:")
display(
    speed_bin_summary[
        [
            "speed_bin",
            "observations",
            "share_of_first_serves_pct",
            "mean_speed",
            "median_speed",
            "server_win_pct"
        ]
    ].round(2)
)

# ------------------------------------------------------------
# Identify modal speed bin
# ------------------------------------------------------------

modal_row = speed_bin_summary.loc[
    speed_bin_summary["observations"].idxmax()
]

print("\nModal first-serve speed bin:")
print(modal_row["speed_bin"])

print(
    "Observations in modal bin:",
    int(modal_row["observations"])
)

print(
    "Share of first serves:",
    round(modal_row["share_of_first_serves_pct"], 2),
    "%"
)

print("\n" + "=" * 70)
print("V9 FIRST-SERVE SPEED-BIN ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 13 — Empirical Speed vs Modeled Admissibility

print("=" * 70)
print("V9 — EMPIRICAL SPEED VS MODELED ADMISSIBILITY")
print("=" * 70)

# ------------------------------------------------------------
# Validated V6 computational results
# Zero spin, 3 m contact height, deuce side,
# azimuth = 0 degrees.
# ------------------------------------------------------------

v6_speed_results = pd.DataFrame({
    "speed_kmh": [
        160, 170, 180, 190, 200, 210, 220
    ],
    "net_boundary_deg": [
        -7.960193,
        -8.186953,
        -8.377152,
        -8.538247,
        -8.675886,
        -8.794410,
        -8.897200
    ],
    "service_boundary_deg": [
        -5.943791,
        -6.326899,
        -6.648231,
        -6.920411,
        -7.152987,
        -7.353293,
        -7.527037
    ],
    "admissible_width_deg": [
        2.016402,
        1.860053,
        1.728921,
        1.617836,
        1.522899,
        1.441117,
        1.370164
    ]
})

# ------------------------------------------------------------
# Empirical first-serve counts at the same speed nodes.
# Use exact 10 km/h bins centered on the V6 speeds.
# ------------------------------------------------------------

empirical_at_model_speeds = []

for speed in v6_speed_results["speed_kmh"]:

    lower = speed - 5
    upper = speed + 5

    subset = first_valid[
        (first_valid["Speed_KMH"] >= lower)
        & (first_valid["Speed_KMH"] < upper)
    ]

    empirical_at_model_speeds.append({
        "speed_kmh": speed,
        "empirical_count": len(subset),
        "empirical_share_pct":
            100 * len(subset) / len(first_valid)
    })

empirical_speed_table = pd.DataFrame(empirical_at_model_speeds)

# ------------------------------------------------------------
# Combine empirical and computational results
# ------------------------------------------------------------

comparison = v6_speed_results.merge(
    empirical_speed_table,
    on="speed_kmh",
    how="left"
)

print("\nComputational admissibility vs observed first-serve population:")
display(comparison.round(4))

# ------------------------------------------------------------
# Overall change across the modeled range
# ------------------------------------------------------------

width_160 = comparison.loc[
    comparison["speed_kmh"] == 160,
    "admissible_width_deg"
].iloc[0]

width_220 = comparison.loc[
    comparison["speed_kmh"] == 220,
    "admissible_width_deg"
].iloc[0]

width_reduction_pct = (
    100 * (width_160 - width_220) / width_160
)

empirical_share_160_220 = (
    first_valid["Speed_KMH"].between(160, 220).mean() * 100
)

print("\nKey results:")
print(
    f"Observed first serves in 160–220 km/h range: "
    f"{empirical_share_160_220:.2f}%"
)

print(
    f"Modeled admissible angular width at 160 km/h: "
    f"{width_160:.4f}°"
)

print(
    f"Modeled admissible angular width at 220 km/h: "
    f"{width_220:.4f}°"
)

print(
    f"Modeled width reduction from 160 to 220 km/h: "
    f"{width_reduction_pct:.2f}%"
)

print("\n" + "=" * 70)
print("V9 EMPIRICAL VS MODELED ADMISSIBILITY COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 13 — Empirical Speed vs Modeled Admissibility
# Corrected labeling of empirical ±5 km/h windows

print("=" * 70)
print("V9 — EMPIRICAL SPEED VS MODELED ADMISSIBILITY")
print("=" * 70)

# ------------------------------------------------------------
# Validated V6 computational results
# Zero spin, 3 m contact height, deuce side,
# azimuth = 0 degrees.
# ------------------------------------------------------------

v6_speed_results = pd.DataFrame({
    "model_speed_kmh": [
        160, 170, 180, 190, 200, 210, 220
    ],
    "net_boundary_deg": [
        -7.960193,
        -8.186953,
        -8.377152,
        -8.538247,
        -8.675886,
        -8.794410,
        -8.897200
    ],
    "service_boundary_deg": [
        -5.943791,
        -6.326899,
        -6.648231,
        -6.920411,
        -7.152987,
        -7.353293,
        -7.527037
    ],
    "admissible_width_deg": [
        2.016402,
        1.860053,
        1.728921,
        1.617836,
        1.522899,
        1.441117,
        1.370164
    ]
})

# ------------------------------------------------------------
# Empirical observations within ±5 km/h of each model speed.
# IMPORTANT:
# These are windows around model nodes, NOT observations
# occurring at exactly the model speed.
# ------------------------------------------------------------

empirical_windows = []

for speed in v6_speed_results["model_speed_kmh"]:

    lower = speed - 5
    upper = speed + 5

    subset = first_valid[
        (first_valid["Speed_KMH"] >= lower)
        & (first_valid["Speed_KMH"] < upper)
    ]

    empirical_windows.append({
        "model_speed_kmh": speed,
        "empirical_window": f"[{lower}, {upper})",
        "observations_within_5_kmh": len(subset),
        "share_of_first_serves_pct":
            100 * len(subset) / len(first_valid)
    })

empirical_window_df = pd.DataFrame(empirical_windows)

# ------------------------------------------------------------
# Combine computational and empirical results
# ------------------------------------------------------------

comparison = v6_speed_results.merge(
    empirical_window_df,
    on="model_speed_kmh",
    how="left"
)

print("\nComputational admissibility vs nearby observed first-serve speeds:")
display(comparison.round(4))

# ------------------------------------------------------------
# Overall empirical coverage of the modeled speed interval
# This is deliberately calculated separately from the ±5 km/h
# windows above.
# ------------------------------------------------------------

observed_160_220 = first_valid[
    first_valid["Speed_KMH"].between(160, 220)
]

empirical_share_160_220 = (
    100 * len(observed_160_220) / len(first_valid)
)

# ------------------------------------------------------------
# Modeled width reduction
# ------------------------------------------------------------

width_160 = v6_speed_results.loc[
    v6_speed_results["model_speed_kmh"] == 160,
    "admissible_width_deg"
].iloc[0]

width_220 = v6_speed_results.loc[
    v6_speed_results["model_speed_kmh"] == 220,
    "admissible_width_deg"
].iloc[0]

width_reduction_pct = (
    100 * (width_160 - width_220) / width_160
)

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\nKey results:")
print(
    f"Observed first serves in 160–220 km/h: "
    f"{len(observed_160_220):,} / {len(first_valid):,}"
)

print(
    f"Observed first serves in 160–220 km/h: "
    f"{empirical_share_160_220:.2f}%"
)

print(
    f"Modeled admissible angular width at 160 km/h: "
    f"{width_160:.4f}°"
)

print(
    f"Modeled admissible angular width at 220 km/h: "
    f"{width_220:.4f}°"
)

print(
    f"Modeled width reduction from 160 to 220 km/h: "
    f"{width_reduction_pct:.2f}%"
)

# ------------------------------------------------------------
# Scientific interpretation
# ------------------------------------------------------------

print("\nInterpretation:")
print(
    "The empirical and computational quantities are complementary: "
    "the dataset describes the observed speed distribution, while "
    "the model describes how admissible launch-angle width changes "
    "with speed under the stated physical assumptions."
)

print("\n" + "=" * 70)
print("V9 EMPIRICAL VS MODELED ADMISSIBILITY — CORRECTED")
print("=" * 70)

In [ ]:
# V9 Cell 14 — Statistical Comparison of First vs Second Serve Speeds

from scipy import stats

print("=" * 70)
print("V9 — FIRST VS SECOND SERVE SPEED COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# Extract valid speed samples
# ------------------------------------------------------------

first_speed = first_valid["Speed_KMH"].to_numpy()
second_speed = second_valid["Speed_KMH"].to_numpy()

# ------------------------------------------------------------
# Basic differences
# ------------------------------------------------------------

mean_difference = first_speed.mean() - second_speed.mean()
median_difference = np.median(first_speed) - np.median(second_speed)

# Pooled standard deviation for standardized mean difference
n1 = len(first_speed)
n2 = len(second_speed)

pooled_sd = np.sqrt(
    (
        (n1 - 1) * np.var(first_speed, ddof=1)
        + (n2 - 1) * np.var(second_speed, ddof=1)
    )
    / (n1 + n2 - 2)
)

cohens_d = mean_difference / pooled_sd

# ------------------------------------------------------------
# Kolmogorov-Smirnov two-sample test
# ------------------------------------------------------------

ks_statistic, ks_pvalue = stats.ks_2samp(
    first_speed,
    second_speed
)

# ------------------------------------------------------------
# Bootstrap confidence interval for median difference
#
# Difference = median(first serve) - median(second serve)
# ------------------------------------------------------------

rng = np.random.default_rng(42)

N_BOOTSTRAP = 5000

bootstrap_differences = np.empty(N_BOOTSTRAP)

for i in range(N_BOOTSTRAP):

    first_sample = rng.choice(
        first_speed,
        size=n1,
        replace=True
    )

    second_sample = rng.choice(
        second_speed,
        size=n2,
        replace=True
    )

    bootstrap_differences[i] = (
        np.median(first_sample)
        - np.median(second_sample)
    )

ci_lower, ci_upper = np.percentile(
    bootstrap_differences,
    [2.5, 97.5]
)

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

comparison_summary = pd.DataFrame({
    "Metric": [
        "First-serve mean",
        "Second-serve mean",
        "Mean difference",
        "First-serve median",
        "Second-serve median",
        "Median difference",
        "95% CI lower — median difference",
        "95% CI upper — median difference",
        "Pooled SD",
        "Cohen's d",
        "KS statistic",
        "KS p-value"
    ],
    "Value": [
        first_speed.mean(),
        second_speed.mean(),
        mean_difference,
        np.median(first_speed),
        np.median(second_speed),
        median_difference,
        ci_lower,
        ci_upper,
        pooled_sd,
        cohens_d,
        ks_statistic,
        ks_pvalue
    ]
})

print("\nStatistical comparison:")
display(comparison_summary.round(4))

# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print("\nInterpretation:")
print(
    f"First serves were {mean_difference:.2f} km/h faster on average "
    f"than second serves."
)

print(
    f"The median difference was {median_difference:.2f} km/h."
)

print(
    f"The bootstrap 95% CI for the median difference was "
    f"[{ci_lower:.2f}, {ci_upper:.2f}] km/h."
)

print(
    f"The KS statistic was {ks_statistic:.4f} "
    f"with p = {ks_pvalue:.3e}."
)

print(
    f"Cohen's d for the mean difference was {cohens_d:.2f}."
)

print("\n" + "=" * 70)
print("V9 FIRST VS SECOND SERVE COMPARISON COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 15 — First-Serve Speed vs Point Outcome
# Corrected: self-contained Wilson CI + scipy optimization

from scipy import optimize

print("=" * 70)
print("V9 — FIRST-SERVE SPEED VS POINT OUTCOME")
print("=" * 70)

# ------------------------------------------------------------
# Prepare first-serve observations
# ------------------------------------------------------------

speed_outcome = first_valid.copy()

speed_outcome["server_won_point"] = (
    speed_outcome["PointWinner"] == speed_outcome["PointServer"]
)

# ------------------------------------------------------------
# Wilson confidence interval
# ------------------------------------------------------------

def wilson_ci(wins, n, z=1.96):

    if n == 0:
        return np.nan, np.nan

    p = wins / n

    denominator = 1 + (z**2 / n)

    center = (
        p + (z**2 / (2 * n))
    ) / denominator

    margin = (
        z
        * np.sqrt(
            (p * (1 - p) / n)
            + (z**2 / (4 * n**2))
        )
        / denominator
    )

    return center - margin, center + margin


# ------------------------------------------------------------
# Create 10 km/h speed bins
# ------------------------------------------------------------

bins = np.arange(90, 241, 10)

speed_outcome["speed_bin"] = pd.cut(
    speed_outcome["Speed_KMH"],
    bins=bins,
    right=False
)

# ------------------------------------------------------------
# Aggregate by speed bin
# ------------------------------------------------------------

rows = []

for interval, group in speed_outcome.groupby(
    "speed_bin",
    observed=False
):

    n = len(group)

    if n == 0:
        continue

    wins = int(group["server_won_point"].sum())

    win_rate = wins / n

    ci_low, ci_high = wilson_ci(
        wins,
        n
    )

    rows.append({
        "speed_bin": str(interval),
        "n": n,
        "server_wins": wins,
        "server_win_pct": 100 * win_rate,
        "ci_low_pct": 100 * ci_low,
        "ci_high_pct": 100 * ci_high
    })

speed_outcome_summary = pd.DataFrame(rows)

print("\nFirst-serve speed vs server point-win rate:")
display(
    speed_outcome_summary.round(2)
)

# ------------------------------------------------------------
# Logistic regression
#
# Outcome:
#   1 = server wins point
#   0 = server loses point
#
# Predictor:
#   speed in km/h
#
# We center speed at 170 km/h to improve numerical stability.
# ------------------------------------------------------------

x_raw = speed_outcome["Speed_KMH"].to_numpy()

x = (
    (x_raw - 170.0) / 10.0
)

y = (
    speed_outcome["server_won_point"]
    .astype(int)
    .to_numpy()
)

# ------------------------------------------------------------
# Negative log-likelihood
# ------------------------------------------------------------

def logistic_nll(beta):

    intercept = beta[0]
    slope = beta[1]

    linear_predictor = (
        intercept + slope * x
    )

    # Stable logistic likelihood
    log_likelihood = (
        y * (-np.logaddexp(0, -linear_predictor))
        +
        (1 - y) * (-np.logaddexp(0, linear_predictor))
    )

    return -np.sum(log_likelihood)


# ------------------------------------------------------------
# Fit model
# ------------------------------------------------------------

result = optimize.minimize(
    logistic_nll,
    x0=np.array([0.8, 0.1]),
    method="BFGS"
)

intercept = result.x[0]
slope_per_10kmh = result.x[1]

# Odds ratio for a 10 km/h increase
odds_ratio_10 = np.exp(slope_per_10kmh)

print("\nLogistic association:")
print(
    f"Intercept at 170 km/h: {intercept:.6f}"
)

print(
    f"Coefficient per 10 km/h: {slope_per_10kmh:.6f}"
)

print(
    f"Odds ratio per 10 km/h: {odds_ratio_10:.4f}"
)

print(
    "Optimization converged:",
    result.success
)

# ------------------------------------------------------------
# Model fit sanity check
# ------------------------------------------------------------

predicted_probability = (
    1 /
    (
        1 +
        np.exp(
            -(intercept + slope_per_10kmh * x)
        )
    )
)

print(
    "\nObserved overall server point-win rate:",
    round(100 * y.mean(), 2),
    "%"
)

print(
    "Mean modeled point-win probability:",
    round(100 * predicted_probability.mean(), 2),
    "%"
)

# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print("\nInterpretation:")
print(
    "The speed-bin results describe the observed association "
    "between recorded first-serve speed and point outcome."
)

print(
    "The logistic model summarizes the direction and magnitude "
    "of that association per 10 km/h increase in recorded speed."
)

print(
    "This is observational and does not establish that increasing "
    "serve speed itself causes a higher probability of winning."
)

print("\n" + "=" * 70)
print("V9 FIRST-SERVE SPEED VS POINT OUTCOME COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 15A — Robust Logistic Regression Fit

from scipy import optimize

print("=" * 70)
print("V9 — ROBUST LOGISTIC ASSOCIATION")
print("=" * 70)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

x_raw = first_valid["Speed_KMH"].to_numpy(dtype=float)

# Center at 170 km/h and scale in 10 km/h units
x = (x_raw - 170.0) / 10.0

y = (
    first_valid["PointWinner"].to_numpy()
    == first_valid["PointServer"].to_numpy()
)

y = y.astype(float)

# ------------------------------------------------------------
# Stable logistic negative log-likelihood
# ------------------------------------------------------------

def logistic_nll(beta):

    eta = beta[0] + beta[1] * x

    return np.sum(
        np.logaddexp(0, eta) - y * eta
    )

# ------------------------------------------------------------
# Analytic gradient
# ------------------------------------------------------------

def logistic_gradient(beta):

    eta = beta[0] + beta[1] * x

    # Stable sigmoid
    probability = np.empty_like(eta)

    positive = eta >= 0
    probability[positive] = (
        1.0 / (1.0 + np.exp(-eta[positive]))
    )

    exp_eta = np.exp(eta[~positive])
    probability[~positive] = (
        exp_eta / (1.0 + exp_eta)
    )

    residual = probability - y

    gradient_intercept = residual.sum()
    gradient_slope = np.sum(residual * x)

    return np.array([
        gradient_intercept,
        gradient_slope
    ])

# ------------------------------------------------------------
# Fit using L-BFGS-B
# ------------------------------------------------------------

initial_beta = np.array([
    0.8,
    0.1
])

result_lbfgs = optimize.minimize(
    logistic_nll,
    initial_beta,
    jac=logistic_gradient,
    method="L-BFGS-B",
    options={
        "ftol": 1e-12,
        "gtol": 1e-10,
        "maxiter": 10000,
        "maxls": 50
    }
)

intercept = result_lbfgs.x[0]
slope_per_10kmh = result_lbfgs.x[1]

odds_ratio_10 = np.exp(slope_per_10kmh)

# ------------------------------------------------------------
# Gradient check at solution
# ------------------------------------------------------------

gradient_at_solution = logistic_gradient(
    result_lbfgs.x
)

gradient_norm = np.linalg.norm(
    gradient_at_solution
)

# ------------------------------------------------------------
# Predicted probability at selected speeds
# ------------------------------------------------------------

def predicted_probability(speed_kmh):

    x_value = (speed_kmh - 170.0) / 10.0

    eta = (
        intercept
        + slope_per_10kmh * x_value
    )

    return 1.0 / (1.0 + np.exp(-eta))


selected_speeds = np.array([
    150, 160, 170, 180, 190, 200, 210, 220
])

predicted_table = pd.DataFrame({
    "speed_kmh": selected_speeds,
    "predicted_server_win_pct": [
        100 * predicted_probability(s)
        for s in selected_speeds
    ]
})

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("\nOptimization result:")
print("Success:", result_lbfgs.success)
print("Status:", result_lbfgs.status)
print("Message:", result_lbfgs.message)

print("\nFitted parameters:")
print(
    f"Intercept at 170 km/h: {intercept:.6f}"
)

print(
    f"Coefficient per 10 km/h: {slope_per_10kmh:.6f}"
)

print(
    f"Odds ratio per 10 km/h: {odds_ratio_10:.4f}"
)

print(
    f"Gradient norm at solution: {gradient_norm:.6e}"
)

print("\nPredicted server point-win probability:")
display(
    predicted_table.round(2)
)

# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

print("\nSanity check:")
print(
    "Observed overall server point-win rate:",
    round(100 * y.mean(), 2),
    "%"
)

print("\nInterpretation:")
print(
    "The logistic model summarizes the observational association "
    "between recorded first-serve speed and point outcome."
)

print(
    "The odds ratio represents the multiplicative change in the "
    "odds of winning the point associated with a 10 km/h increase "
    "in recorded first-serve speed."
)

print(
    "This association is not causal and may reflect confounding "
    "from player ability, placement, opponent quality, serve type, "
    "or other unmeasured variables."
)

print("\n" + "=" * 70)
print("V9 ROBUST LOGISTIC ASSOCIATION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 16 — Test Linear vs Quadratic Speed Association

from scipy import optimize

print("=" * 70)
print("V9 — LINEAR VS QUADRATIC SPEED ASSOCIATION")
print("=" * 70)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

x_raw = first_valid["Speed_KMH"].to_numpy(dtype=float)

# Center and scale speed
x = (x_raw - 170.0) / 10.0

y = (
    first_valid["PointWinner"].to_numpy()
    == first_valid["PointServer"].to_numpy()
)

y = y.astype(float)

# ------------------------------------------------------------
# Logistic negative log-likelihood
# ------------------------------------------------------------

def logistic_nll_linear(beta):

    eta = beta[0] + beta[1] * x

    return np.sum(
        np.logaddexp(0, eta) - y * eta
    )


def logistic_nll_quadratic(beta):

    eta = (
        beta[0]
        + beta[1] * x
        + beta[2] * x**2
    )

    return np.sum(
        np.logaddexp(0, eta) - y * eta
    )

# ------------------------------------------------------------
# Fit linear model
# ------------------------------------------------------------

linear_result = optimize.minimize(
    logistic_nll_linear,
    x0=np.array([0.8, 0.1]),
    method="L-BFGS-B",
    options={
        "ftol": 1e-12,
        "gtol": 1e-10,
        "maxiter": 10000
    }
)

# ------------------------------------------------------------
# Fit quadratic model
# ------------------------------------------------------------

quadratic_result = optimize.minimize(
    logistic_nll_quadratic,
    x0=np.array([0.8, 0.1, 0.0]),
    method="L-BFGS-B",
    options={
        "ftol": 1e-12,
        "gtol": 1e-10,
        "maxiter": 10000
    }
)

# ------------------------------------------------------------
# Calculate AIC
# ------------------------------------------------------------

n = len(y)

linear_nll = linear_result.fun
quadratic_nll = quadratic_result.fun

linear_aic = (
    2 * 2
    + 2 * linear_nll
)

quadratic_aic = (
    2 * 3
    + 2 * quadratic_nll
)

# ------------------------------------------------------------
# Likelihood-ratio statistic
# ------------------------------------------------------------

likelihood_ratio = (
    2 * (linear_nll - quadratic_nll)
)

# One additional parameter
lr_pvalue = 1 - stats.chi2.cdf(
    likelihood_ratio,
    df=1
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("\nLinear model:")
print("Converged:", linear_result.success)
print("Intercept:", linear_result.x[0])
print("Linear coefficient:", linear_result.x[1])
print("NLL:", linear_nll)
print("AIC:", linear_aic)

print("\nQuadratic model:")
print("Converged:", quadratic_result.success)
print("Intercept:", quadratic_result.x[0])
print("Linear coefficient:", quadratic_result.x[1])
print("Quadratic coefficient:", quadratic_result.x[2])
print("NLL:", quadratic_nll)
print("AIC:", quadratic_aic)

print("\nModel comparison:")
print("Likelihood-ratio statistic:", likelihood_ratio)
print("Likelihood-ratio p-value:", lr_pvalue)
print("AIC difference (quadratic - linear):",
      quadratic_aic - linear_aic)

# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

if quadratic_aic < linear_aic:
    preferred = "quadratic"
else:
    preferred = "linear"

print("\nPreferred model by AIC:", preferred)

print("\nInterpretation:")
print(
    "The comparison tests whether adding a quadratic speed term "
    "provides enough improvement in fit to justify the additional "
    "parameter."
)

print(
    "This is an empirical model-selection exercise and does not "
    "imply that the underlying serve physics is quadratic."
)

print("\n" + "=" * 70)
print("V9 LINEAR VS QUADRATIC SPEED ASSOCIATION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 17 — Robustness of Linear vs Quadratic Speed Association
# Restrict analysis to the well-populated 120–220 km/h range.

from scipy import optimize, stats

print("=" * 70)
print("V9 — ROBUSTNESS OF SPEED-OUTCOME FUNCTIONAL FORM")
print("=" * 70)

# ------------------------------------------------------------
# Restrict to 120–220 km/h
# ------------------------------------------------------------

robust_sample = first_valid[
    first_valid["Speed_KMH"].between(120, 220)
].copy()

x_raw = robust_sample["Speed_KMH"].to_numpy(dtype=float)

x = (x_raw - 170.0) / 10.0

y = (
    robust_sample["PointWinner"].to_numpy()
    == robust_sample["PointServer"].to_numpy()
)

y = y.astype(float)

print("\nRestricted sample:")
print("Speed range: 120–220 km/h")
print("N:", len(y))

# ------------------------------------------------------------
# Negative log-likelihoods
# ------------------------------------------------------------

def nll_linear(beta):

    eta = beta[0] + beta[1] * x

    return np.sum(
        np.logaddexp(0, eta) - y * eta
    )


def nll_quadratic(beta):

    eta = (
        beta[0]
        + beta[1] * x
        + beta[2] * x**2
    )

    return np.sum(
        np.logaddexp(0, eta) - y * eta
    )

# ------------------------------------------------------------
# Fit linear model
# ------------------------------------------------------------

linear_fit = optimize.minimize(
    nll_linear,
    x0=np.array([0.8, 0.1]),
    method="L-BFGS-B",
    options={
        "ftol": 1e-12,
        "gtol": 1e-10,
        "maxiter": 10000
    }
)

# ------------------------------------------------------------
# Fit quadratic model
# ------------------------------------------------------------

quadratic_fit = optimize.minimize(
    nll_quadratic,
    x0=np.array([0.8, 0.1, 0.0]),
    method="L-BFGS-B",
    options={
        "ftol": 1e-12,
        "gtol": 1e-10,
        "maxiter": 10000
    }
)

# ------------------------------------------------------------
# AIC
# ------------------------------------------------------------

linear_aic = (
    2 * 2
    + 2 * linear_fit.fun
)

quadratic_aic = (
    2 * 3
    + 2 * quadratic_fit.fun
)

delta_aic = quadratic_aic - linear_aic

# ------------------------------------------------------------
# Likelihood-ratio test
# ------------------------------------------------------------

lr_stat = (
    2 * (linear_fit.fun - quadratic_fit.fun)
)

lr_pvalue = 1 - stats.chi2.cdf(
    lr_stat,
    df=1
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("\nLinear model:")
print("Converged:", linear_fit.success)
print("Intercept:", linear_fit.x[0])
print("Linear coefficient:", linear_fit.x[1])
print("AIC:", linear_aic)

print("\nQuadratic model:")
print("Converged:", quadratic_fit.success)
print("Intercept:", quadratic_fit.x[0])
print("Linear coefficient:", quadratic_fit.x[1])
print("Quadratic coefficient:", quadratic_fit.x[2])
print("AIC:", quadratic_aic)

print("\nModel comparison:")
print("Delta AIC (quadratic - linear):", delta_aic)
print("Likelihood-ratio statistic:", lr_stat)
print("Likelihood-ratio p-value:", lr_pvalue)

if quadratic_aic < linear_aic:
    print("\nPreferred model by AIC: QUADRATIC")
else:
    print("\nPreferred model by AIC: LINEAR")

print("\nInterpretation:")
print(
    "This robustness analysis tests whether the functional-form "
    "preference remains after excluding sparsely populated extreme "
    "speed observations."
)

print(
    "A persistent quadratic preference would indicate that the "
    "observed curvature is not primarily driven by the extreme "
    "high-speed tail."
)

print("\n" + "=" * 70)
print("V9 SPEED-OUTCOME FUNCTIONAL FORM ROBUSTNESS COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 18 — Empirical Cumulative Speed Distribution

print("=" * 70)
print("V9 — EMPIRICAL CUMULATIVE SPEED DISTRIBUTION")
print("=" * 70)

# ------------------------------------------------------------
# V6 computational speed nodes
# ------------------------------------------------------------

speed_nodes = np.array([
    160, 170, 180, 190, 200, 210, 220
])

# ------------------------------------------------------------
# Calculate empirical cumulative distribution
# ------------------------------------------------------------

cdf_rows = []

for speed in speed_nodes:

    count_at_or_below = (
        first_valid["Speed_KMH"] <= speed
    ).sum()

    count_above = (
        first_valid["Speed_KMH"] > speed
    ).sum()

    cdf_rows.append({
        "speed_kmh": speed,
        "first_serves_at_or_below": count_at_or_below,
        "first_serves_above": count_above,
        "pct_at_or_below": (
            100 * count_at_or_below / len(first_valid)
        ),
        "pct_above": (
            100 * count_above / len(first_valid)
        )
    })

empirical_cdf = pd.DataFrame(cdf_rows)

print("\nEmpirical cumulative distribution:")
display(
    empirical_cdf.round(2)
)

# ------------------------------------------------------------
# Add V6 admissibility width
# ------------------------------------------------------------

v6_widths = pd.DataFrame({
    "speed_kmh": [
        160, 170, 180, 190, 200, 210, 220
    ],
    "admissible_width_deg": [
        2.016402,
        1.860053,
        1.728921,
        1.617836,
        1.522899,
        1.441117,
        1.370164
    ]
})

combined_cdf = empirical_cdf.merge(
    v6_widths,
    on="speed_kmh",
    how="left"
)

print("\nEmpirical distribution + computational admissibility:")
display(
    combined_cdf.round(4)
)

# ------------------------------------------------------------
# Key empirical quantiles
# ------------------------------------------------------------

print("\nKey first-serve speed percentiles:")

for percentile in [10, 25, 50, 75, 90, 95, 99]:

    value = first_valid["Speed_KMH"].quantile(
        percentile / 100
    )

    print(
        f"{percentile}th percentile: "
        f"{value:.1f} km/h"
    )

print("\n" + "=" * 70)
print("V9 EMPIRICAL CUMULATIVE SPEED DISTRIBUTION COMPLETE")
print("=" * 70)

In [ ]:
# V9 Cell 19 — Final Empirical/Computational Summary

print("=" * 70)
print("V9 — FINAL EMPIRICAL / COMPUTATIONAL SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# Core empirical statistics
# ------------------------------------------------------------

first_n = len(first_valid)
second_n = len(second_valid)

first_mean = first_valid["Speed_KMH"].mean()
second_mean = second_valid["Speed_KMH"].mean()

first_median = first_valid["Speed_KMH"].median()
second_median = second_valid["Speed_KMH"].median()

mean_difference = first_mean - second_mean
median_difference = first_median - second_median

# ------------------------------------------------------------
# First-serve speed coverage
# ------------------------------------------------------------

first_160_220 = first_valid[
    first_valid["Speed_KMH"].between(160, 220)
]

coverage_160_220 = (
    100 * len(first_160_220) / first_n
)

pct_at_200 = (
    100
    * (first_valid["Speed_KMH"] <= 200).sum()
    / first_n
)

pct_above_200 = (
    100
    * (first_valid["Speed_KMH"] > 200).sum()
    / first_n
)

pct_at_210 = (
    100
    * (first_valid["Speed_KMH"] <= 210).sum()
    / first_n
)

# ------------------------------------------------------------
# Computational envelope
# ------------------------------------------------------------

width_160 = 2.016402
width_200 = 1.522899
width_220 = 1.370164

width_reduction = (
    100 * (width_160 - width_220) / width_160
)

# ------------------------------------------------------------
# Speed / outcome association
# ------------------------------------------------------------

odds_ratio_10 = 1.1347

# ------------------------------------------------------------
# Build summary table
# ------------------------------------------------------------

summary = pd.DataFrame({
    "Quantity": [
        "Valid first-serve speeds",
        "Valid second-serve speeds",
        "First-serve mean speed",
        "Second-serve mean speed",
        "First–second mean difference",
        "First-serve median speed",
        "Second-serve median speed",
        "First–second median difference",
        "First serves in 160–220 km/h",
        "First serves at or below 200 km/h",
        "First serves above 200 km/h",
        "First serves at or below 210 km/h",
        "Modeled width at 160 km/h",
        "Modeled width at 200 km/h",
        "Modeled width at 220 km/h",
        "Modeled width reduction, 160→220",
        "Odds ratio per 10 km/h"
    ],
    "Value": [
        first_n,
        second_n,
        first_mean,
        second_mean,
        mean_difference,
        first_median,
        second_median,
        median_difference,
        coverage_160_220,
        pct_at_200,
        pct_above_200,
        pct_at_210,
        width_160,
        width_200,
        width_220,
        width_reduction,
        odds_ratio_10
    ],
    "Units": [
        "observations",
        "observations",
        "km/h",
        "km/h",
        "km/h",
        "km/h",
        "km/h",
        "km/h",
        "%",
        "%",
        "%",
        "%",
        "degrees",
        "degrees",
        "degrees",
        "%",
        "odds ratio"
    ]
})

print("\nFinal V9 summary:")
display(summary)

# ------------------------------------------------------------
# Scientific conclusions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V9 SCIENTIFIC CONCLUSIONS")
print("=" * 70)

print(
    "\n1. The 2024 US Open dataset contains a large empirical sample "
    f"of {first_n:,} valid first-serve and {second_n:,} valid "
    "second-serve speed observations."
)

print(
    f"\n2. First serves were substantially faster than second serves: "
    f"{mean_difference:.2f} km/h higher in mean speed and "
    f"{median_difference:.0f} km/h higher in median speed."
)

print(
    f"\n3. {coverage_160_220:.2f}% of valid first-serve speeds "
    "fell within the 160–220 km/h range evaluated computationally."
)

print(
    f"\n4. The computational admissible angular width decreased "
    f"from {width_160:.4f}° at 160 km/h to "
    f"{width_220:.4f}° at 220 km/h, "
    f"a {width_reduction:.2f}% reduction."
)

print(
    f"\n5. {pct_at_200:.2f}% of observed first serves were "
    "at or below 200 km/h, placing 200 km/h in the upper "
    "portion of the observed speed distribution."
)

print(
    f"\n6. Recorded first-serve speed showed a positive observational "
    f"association with server point outcome, with an estimated "
    f"odds ratio of {odds_ratio_10:.4f} per 10 km/h increase."
)

print(
    "\n7. These empirical results do not establish causality and "
    "do not identify the launch-angle, spin, or contact-point "
    "strategies used by individual players."
)

print(
    "\n8. The computational and empirical analyses are therefore "
    "interpreted as complementary: the model quantifies physical "
    "admissibility, while the tournament data characterize the "
    "observed speed regime."
)

print("\n" + "=" * 70)
print("V9 COMPLETE — EMPIRICAL VALIDATION SUMMARY READY")
print("=" * 70)